In [9]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Using project root:", project_root)
print("Ensured project root on sys.path")

Using project root: /Users/lisavanoosten/Desktop/Big-Data/big_data_assignment
Ensured project root on sys.path


### Environment setup

Before running this notebook, install the project dependencies in a terminal from the project root (`big_data_assignment`):

```bash
cd ~/Desktop/Big-Data/big_data_assignment
pip install -r requirements.txt
```

After installing, restart the Jupyter kernel and re-run the first environment cell.

# IMDB DuckDB Pipeline: Movies, Directors, and Writers

This notebook sets up a **reusable DuckDB pipeline** for the IMDB project:

- Reads IMDB CSV and JSON metadata from `big-data-course-2024-projects/imdb/`.
- Normalizes directors and writers JSON into many-to-many **edge tables**:
  - `imdb_directing_edges(tconst, director_id)`
  - `imdb_writing_edges(tconst, writer_id)`
- Shows how to join these edges back to the main train data.


In [6]:
import duckdb
from pathlib import Path

# Paths
base_path = Path("..") / "big-data-course-2024-projects" / "imdb"

con = duckdb.connect(database="imdb.duckdb", read_only=False)
con

# Create / refresh main train, directing, and writing raw tables
con.execute(
    """
    CREATE OR REPLACE TABLE imdb_train AS
    SELECT *
    FROM read_csv_auto(? || '/train-*.csv', HEADER=TRUE);
    """,
    [str(base_path)],
)

con.execute(
    """
    CREATE OR REPLACE TABLE imdb_directing_raw AS
    SELECT * FROM read_json_auto(? || '/directing.json');
    """,
    [str(base_path)],
)

con.execute(
    """
    CREATE OR REPLACE TABLE imdb_writing_raw AS
    SELECT * FROM read_json_auto(? || '/writing.json');
    """,
    [str(base_path)],
)

print("Rows in imdb_train:", con.execute("SELECT COUNT(*) FROM imdb_train").fetchone()[0])
print("Rows in imdb_directing_raw:", con.execute("SELECT COUNT(*) FROM imdb_directing_raw").fetchone()[0])
print("Rows in imdb_writing_raw:", con.execute("SELECT COUNT(*) FROM imdb_writing_raw").fetchone()[0])


IOException: IO Error: No files found that match the pattern "../big-data-course-2024-projects/imdb/train-*.csv"

LINE 4:     FROM read_csv_auto(? || '/train-*.csv', HEADER=TRUE);
                 ^

In [7]:
# Deprecated cell (old attempt using UNNEST/map_entries).
# The actual edge tables are built in the next cell (
# "Build directing/writing edge tables without UNNEST").
# You can safely skip running this cell.

print("Skip this cell: use the next cell to build imdb_directing_edges and imdb_writing_edges.")


Skip this cell: use the next cell to build imdb_directing_edges and imdb_writing_edges.


In [8]:
# Build directing/writing edge tables without UNNEST (compatible with current JSON schema)

import pandas as pd

# 1) Build imdb_directing_edges from the single-row JSON table
raw_df = con.execute("SELECT movie, director FROM imdb_directing_raw").fetchdf()
row = raw_df.iloc[0]
movie_map = row["movie"]          # dict-like: index -> tconst
director_map = row["director"]    # dict-like: index -> director_id

pairs = [
    (movie_map[k], director_map[k])
    for k in movie_map.keys()
    if k in director_map
]

directors_df = pd.DataFrame(pairs, columns=["tconst", "director_id"])
con.register("directors_df", directors_df)
con.execute("CREATE OR REPLACE TABLE imdb_directing_edges AS SELECT * FROM directors_df")

# 2) Build imdb_writing_edges directly from imdb_writing_raw (already one row per movie-writer)
con.execute(
    """
    CREATE OR REPLACE TABLE imdb_writing_edges AS
    SELECT
      movie::VARCHAR AS tconst,
      writer        AS writer_id
    FROM imdb_writing_raw;
    """
)

print("Example edges (directors):")
display(con.execute("SELECT * FROM imdb_directing_edges LIMIT 10").fetchdf())

print("Example edges (writers):")
display(con.execute("SELECT * FROM imdb_writing_edges LIMIT 10").fetchdf())

CatalogException: Catalog Error: Table with name imdb_directing_raw does not exist!
Did you mean "pg_description"?

LINE 1: SELECT movie, director FROM imdb_directing_raw
                                    ^

In [7]:
# Example: join edges back to train to get directors and writers per movie

query = """
WITH train AS (
  SELECT * FROM imdb_train
)
SELECT
  t.tconst,
  t.primaryTitle,
  list_agg(DISTINCT d.director_id) AS director_ids,
  list_agg(DISTINCT w.writer_id)   AS writer_ids
FROM train t
LEFT JOIN imdb_directing_edges d USING (tconst)
LEFT JOIN imdb_writing_edges   w USING (tconst)
GROUP BY t.tconst, t.primaryTitle
ORDER BY t.tconst
LIMIT 20;
"""

df_example = con.execute(query).fetchdf()
df_example


CatalogException: Catalog Error: Table with name imdb_directing_edges does not exist!
Did you mean "imdb_directing"?

LINE 11: LEFT JOIN imdb_directing_edges d USING (tconst)
                   ^